In [7]:
%pip install -q pymupdf langchain-community langchain-classic langchain-text-splitters langchain-google-genai langchain-groq python-dotenv chromadb sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [8]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_groq import ChatGroq
from langchain_classic.chains import RetrievalQA
from langchain_community.vectorstores import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings

In [16]:
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")
if groq_api_key:
    os.environ["GROQ_API_KEY"] = groq_api_key

gemini_api_key = os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")
if gemini_api_key:
    os.environ["GOOGLE_API_KEY"] = gemini_api_key
    os.environ["GEMINI_API_KEY"] = gemini_api_key

In [10]:
## 1. Load the PDF document
def load_pdf(path):
    loader = PyMuPDFLoader(path)
    return loader.load()

pdf_path = os.path.join("document_rag", "attention-is-all-you-need.pdf")
if not os.path.isfile(pdf_path):
    pdf_path = "attention-is-all-you-need.pdf"

documents = load_pdf(pdf_path)

In [11]:
## 2. Split the document into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
docs = text_splitter.split_documents(documents)

In [12]:
## 3. Create embeddings
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en")

C:\Users\Speeker\AppData\Local\Temp\ipykernel_6680\185347094.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en")
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7806.53it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
## 4. Store in Chroma DB
vectorstore = Chroma.from_documents(docs, embeddings, persist_directory="./pdf_rag_data")
vectorstore.persist()

C:\Users\Speeker\AppData\Local\Temp\ipykernel_6680\2850087926.py:3: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()


In [14]:
!pip install langchain-google-genai

Defaulting to user installation because normal site-packages is not writeable


In [17]:
from google import genai
from langchain_google_genai import ChatGoogleGenerativeAI


llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.1,
    max_output_tokens=512,
    api_key=gemini_api_key,
)
            

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [18]:
## 6. Create the RAG QA chain
rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),
    return_source_documents=True
)

In [19]:
## 7. Ask Some questions
questions = [
    "What is the paper about?",
    "How Many layers are there in the Transformer model?",
    "What is the role of attention in the Transformer model?",
    "What is the number is Encodes and Decodes in the Transformer model?"
    "What is the main contribution of the paper?",
    "What are the key components of the Transformer architecture?",
    
]

In [21]:
for q in questions:
    result = rag_chain({"query": q})
    print(f"\nQ: {q}")
    print(f"A: {result['result']}")
    print("---------------------------------------------------------------------")


Q: What is the paper about?
A: The paper is about evaluating the importance of different components of the Transformer architecture. It does this by varying the base model in different ways and measuring the change in performance on English-to-German translation.
---------------------------------------------------------------------

Q: How Many layers are there in the Transformer model?
A: The Transformer model has an encoder composed of 6 identical layers and a decoder also composed of 6 identical layers.
---------------------------------------------------------------------

Q: What is the role of attention in the Transformer model?
A: The Transformer model uses multi-head attention in several key ways:

1.  **Addressing Long-
---------------------------------------------------------------------

Q: What is the number is Encodes and Decodes in the Transformer model?What is the main contribution of the paper?
A: Based on the provided text:

1.  **Number of Encoders and Decoders:** The